# Snow water in river basins: ERA5-Land SWE, precipitation as snow, and onset anomalies

Per-basin April-1 (Oct-1 in the south) SWE from ERA5-Land, the long-term share of precipitation
falling as snow, basin water volumes, and their relation to the runoff-onset anomalies; plus the
basin-mean spring temperature anomaly from the cube's ERA5 zonal means. Writes
`results/<version>/river_basin_snow_water.csv` (read by `basin_onset.ipynb`). Needs Earth Engine.

In [ ]:
import geopandas as gpd
import matplotlib
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray as rxr
import seaborn as sns
import xarray as xr
from cartopy import crs as ccrs
from cartopy import feature as cfeature

from gsro_analysis import aggregate, paths, settings, stats
import ee
import easysnowdata
from gsro_analysis.results import save_result_table   # stamps _git_sha (production) and _analysis_git_sha (this repo)

settings.initialize_earthengine()

In [ ]:
config = settings.load_config()  # the dataset version lives in settings.CONFIG_FILE

# the river-basin cube: basin x elevation x chili_class x water_year (+ ERA5 anomaly zonal means)
basins_ds = aggregate.open_aggregate('river_basins', config.version)
basins_ds

In [ ]:
# HydroBASINS level 5 from the locally cached BasinATLAS gdb (the pipeline stores level-6 ids and
# the default river_basins cube is their level-5 prefix, so these are the cube's polygons; the gdb
# spells the reserved word ORDER with a trailing underscore)
basins_gdf = gpd.read_file(
    settings.cached_source(settings.BASIN_ATLAS_URL, filename='BasinATLAS_Data_v10.gdb.zip',
                           expected_md5=settings.BASIN_ATLAS_MD5),
    layer=settings.basin_atlas_layer(5)).rename(columns={'ORDER_': 'ORDER'})
basin_populations_gdf = gpd.read_file(paths.GEOMETRIES / 'Hydrobasins_L5_Population_Global.geojson')
print('total population in all basins (billion):', basin_populations_gdf['total_population'].sum() / 1e9)

# one row per basin: geometry, population, pixel-weighted means of median onset / MAD / yearly
# onset and anomaly, pixel counts and the mapped share of the basin area (means masked where
# < 5 % of the basin is mapped, yearly values where < 1 %)
basins_means_gdf = stats.basin_summary(basins_ds, basins_gdf, basin_populations_gdf)
basins_means_gdf

## ERA5-Land SWE on April 1 (NH) / October 1 (SH) per water year

In [ ]:
hemisphere = "NH" # "NH" or "SH"
water_year = 2015
water_years = [int(y) for y in config.water_years]

datasets = []

for water_year in water_years:

    april1_date = f"{water_year}-04-01"
    nh_bbox_input = (-180,0,180,90)

    era5_land_swe_nh_da = easysnowdata.hydroclimatology.get_era5(
                                                            version="ERA5_LAND",
                                                            bbox_input=nh_bbox_input,
                                                            cadence="DAILY",
                                                            start_date=april1_date,
                                                            end_date=april1_date,
                                                            initialize_ee=False,
                                                            variables="snow_depth_water_equivalent")["snow_depth_water_equivalent"]

    era5_land_swe_nh_da = era5_land_swe_nh_da.assign_coords(
        water_year=era5_land_swe_nh_da.time.dt.year
    ).swap_dims(
        {"time": "water_year"}
    ).drop_vars("time")


    october1_date = f"{water_year}-10-01"
    sh_bbox_input = (-180,-90,180,0)

    era5_land_swe_sh_da = easysnowdata.hydroclimatology.get_era5(
                                                            version="ERA5_LAND",
                                                            bbox_input=sh_bbox_input,
                                                            cadence="DAILY",
                                                            start_date=october1_date,
                                                            end_date=october1_date,
                                                            initialize_ee=False,
                                                            variables="snow_depth_water_equivalent")["snow_depth_water_equivalent"]

    era5_land_swe_sh_da = era5_land_swe_sh_da.assign_coords(
        water_year=era5_land_swe_sh_da.time.dt.year
    ).swap_dims(
        {"time": "water_year"}
    ).drop_vars("time")

    datasets.append(xr.merge([era5_land_swe_nh_da, era5_land_swe_sh_da])["snow_depth_water_equivalent"].squeeze())

era5_land_swe_da = xr.concat(datasets, dim='water_year')
era5_land_swe_da = era5_land_swe_da.where(lambda x: x >= 0)
era5_land_swe_da

In [ ]:
era5_land_swe_da.plot.imshow(col='water_year', col_wrap=5, vmin=0, vmax=1, cmap='Blues', add_colorbar=True, cbar_kwargs={'label': 'SWE [m]'})

In [ ]:
era5_land_swe_median_da = era5_land_swe_da.median(dim='water_year')
era5_land_swe_median_da

In [ ]:
era5_land_swe_anomaly_da = era5_land_swe_da - era5_land_swe_median_da
era5_land_swe_anomaly_da.plot.imshow(col='water_year', col_wrap=2, aspect=2, add_colorbar=True, vmin=-0.4, vmax=0.4, cmap='RdBu', cbar_kwargs={'label': 'SWE anomaly [m]'})

In [ ]:
era5_land_swe_pct_norm_da = 100*(era5_land_swe_da/era5_land_swe_median_da)
era5_land_swe_pct_norm_da.plot.imshow(col='water_year', col_wrap=2, aspect=2, add_colorbar=True, vmin=50, vmax=150, cmap='RdBu', cbar_kwargs={'label': 'SWE pct norm [%]'})

In [ ]:
R = 6.371e6

dϕ = np.deg2rad(0.1)
dλ = np.deg2rad(0.1)

dlat = R * dϕ * xr.ones_like(era5_land_swe_da['longitude'])
dlon = R * dλ * np.cos(np.deg2rad(era5_land_swe_da['latitude']))
dlon.name = "dlon"
dlat.name = "dlat"

cell_area_m_da = dlon * dlat
cell_area_m_da=cell_area_m_da.rio.write_crs(era5_land_swe_da.rio.crs)
cell_area_km_da = cell_area_m_da / 1e6

surface_area = cell_area_km_da.sum()
print(f"Total surface area of the dataset: {surface_area.values:,.0f} km²")


cell_area_km_da

## Long-term share of precipitation falling as snow (ERA5-Land monthly aggregates, 1950 onward)

In [ ]:
# Create yearly sums ImageCollection as before
variables = ["snowfall_sum", "total_precipitation_sum"]
image_collection = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR").select(variables)

start_year = 1950
end_year = 2024

def process_year(year):
    start_date = ee.Date.fromYMD(year, 1, 1)
    end_date = ee.Date.fromYMD(year, 12, 31)
    
    # Filter images for this year
    yearly_collection = image_collection.filterDate(start_date, end_date)
    
    # Sum monthly values to get annual totals
    yearly_sum = yearly_collection.sum()
    
    # Add year as a property
    return yearly_sum.set('year', year)

# Generate list of years and map the function over each year
years = range(start_year, end_year + 1)
yearly_sums = ee.ImageCollection(list(map(lambda y: process_year(y), years)))


mean_image = yearly_sums.mean()

snowfall = mean_image.select('snowfall_sum')
total_precip = mean_image.select('total_precipitation_sum')

valid_precip = total_precip.gt(0)  # Mask where precip > 0
pct_snow = snowfall.divide(total_precip).multiply(100).updateMask(valid_precip)
pct_snow = pct_snow.reproject(crs=image_collection.first().projection())
pct_snow = pct_snow.rename('percent_snow')


# xee >= 0.1 wants the output grid explicitly (crs, crs_transform, shape_2d): the collection's native
# 0.1 deg grid. The static image gets a system:time_start so xee builds its (dropped) time axis quietly.
from xee.helpers import extract_grid_params
grid = extract_grid_params(image_collection)
pct_precip_as_snow_da = xr.open_dataset(
    ee.ImageCollection([pct_snow.set('system:time_start', 0)]), engine='ee', **grid,
)['percent_snow'].isel(time=0, drop=True).compute()

pct_precip_as_snow_da

In [ ]:
pct_precip_as_snow_da = (pct_precip_as_snow_da            # xee 0.1 returns (y, x), north-down
        .rename({'y': 'latitude', 'x': 'longitude'})
        .rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')
        .rio.write_crs(grid['crs']))
pct_precip_as_snow_da

In [ ]:
f,ax=plt.subplots(figsize=(12,6), subplot_kw={'projection': ccrs.Robinson()}, dpi=300, layout='constrained')

# Plot the data
# Define the color scheme similar to the one in the image
# Define the color scheme
colors = ['#7F00FF', '#5000FF', '#2E00FF', '#0000FF', '#003AFF', '#0075FF', 
          '#00AFFF', '#00C8C8', '#00E282', '#37FF00', '#69FF00', '#A0FF00',
          '#D7FF00', '#FFDC00', '#FFA500', '#FF6B00', '#FF3200', '#FF0000', '#FF0080']

# Create custom colormap
cmap = matplotlib.colors.LinearSegmentedColormap.from_list('custom_cmap', colors)
cmap.set_under('white')  # This sets values below vmin to white

# Set the levels for the color scale
levels = np.arange(0, 105, 5)  # 0, 5, 10, 15, ..., 95


plot = pct_precip_as_snow_da.plot(ax=ax, transform=ccrs.PlateCarree(),
               #cmap=cmap, levels=levels, 
               cmap='Purples',
               cbar_kwargs={'label': 'Percentage [%]'},
               add_colorbar=True)

# gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
# gl.top_labels = False
# gl.right_labels = False

ax.coastlines()

ax.set_title('Percentage of annual water-equivalent precipitation that falls as snow')

## Per-basin join + results table

In [ ]:
# for each basin, add SWE median, SWE for each water year, and SWE anomaly for each water year from the era5_land_swe_da cropped to the basin

for i,basin in enumerate(basins_means_gdf['PFAF_ID'].values):
    print(f"Processing basin {basin}... {i+1}/{len(basins_means_gdf['PFAF_ID'].values)}")
    try:
        basin_swe_da = era5_land_swe_da.rio.clip(basins_means_gdf[basins_means_gdf['PFAF_ID'] == basin].geometry.values, crs=basins_means_gdf.crs)
        basin_swe_median_da = basin_swe_da.median(dim='water_year')
        basin_swe_anomaly_da = basin_swe_da - basin_swe_median_da
        basin_swe_pct_norm_da = 100*(basin_swe_da/basin_swe_median_da)

        basin_pct_precip_as_snow_da = pct_precip_as_snow_da.rio.clip(basins_means_gdf[basins_means_gdf['PFAF_ID'] == basin].geometry.values, crs=basins_means_gdf.crs)

        basin_cell_area_da = cell_area_km_da.rio.clip(basins_means_gdf[basins_means_gdf['PFAF_ID'] == basin].geometry.values, crs=basins_means_gdf.crs)
        basin_water_volume_da = ((basin_swe_da/1000) * basin_cell_area_da) # convert to km3
        basin_median_water_volume_da = basin_water_volume_da.median(dim='water_year')
        basin_water_volume_anomaly_da = basin_water_volume_da - basin_median_water_volume_da


    #   basin_swe_sum_da = basin_swe_da.sum(dim='water_year') some sort of sum and median sum (total water from snow)

        basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, 'pct_precip_as_snow'] = basin_pct_precip_as_snow_da.where(lambda x: x>0).mean().values
        basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, 'swe_median'] = basin_swe_median_da.mean().values
        basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, 'median_total_water_equivalent_km3'] = basin_median_water_volume_da.mean().values


        for water_year in water_years:
            basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, f'swe_WY{water_year}'] = basin_swe_da.sel(water_year=water_year).mean().values
            basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, f'swe_anomaly_WY{water_year}'] = basin_swe_anomaly_da.sel(water_year=water_year).mean().values
            basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, f'swe_pct_norm_WY{water_year}'] = basin_swe_pct_norm_da.sel(water_year=water_year).mean().values

            basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, f'total_water_equivalent_km3_WY{water_year}'] = basin_water_volume_da.sel(water_year=water_year).mean().values
            basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, f'total_water_equivalent_anomaly_km3_WY{water_year}'] = basin_water_volume_anomaly_da.sel(water_year=water_year).mean().values
            basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, f'total_water_equivalent_pct_norm_WY{water_year}'] = (basin_water_volume_da.sel(water_year=water_year)/basin_median_water_volume_da).mean().values
    except Exception as e:
        print(f"Error processing basin {basin}: {e}")
        continue

In [ ]:
cols = ['PFAF_ID', 'pct_precip_as_snow', 'swe_median', 'median_total_water_equivalent_km3'] + \
       [c for c in basins_means_gdf.columns if c.startswith(('swe_', 'total_water_equivalent'))]
save_result_table(basins_means_gdf[cols].round(4), 'river_basin_snow_water',
                  results_dir=paths.resultsdir('river_basins', config.version))

In [ ]:
basins_means_gdf.plot(column='swe_median',legend=True, cmap='Blues', legend_kwds={'label': "SWE median [m]"},vmin=0,vmax=1)

In [ ]:
basins_means_gdf.plot(column='pct_precip_as_snow',legend=True, cmap='Purples', legend_kwds={'label': "Percent of precip that falls as snow [%]"},vmin=0,vmax=100)

## Basin-mean spring temperature anomaly (cube ERA5 zonal means) vs onset anomaly

In [ ]:
if 'temperature_2m' in basins_ds:
    spring_t = basins_ds['temperature_2m'].sel(month=stats.SPRING_MONTHS).mean('month')          # (basin, water_year)
    onset_anom = aggregate.weighted_mean(basins_ds, 'runoff_onset_anomaly', ['elevation', 'chili_class'])
    pairs = xr.Dataset({'spring_temp_anomaly': spring_t, 'onset_anomaly': onset_anom}).to_dataframe().dropna()
    f, ax = plt.subplots(figsize=(7, 7))
    hb = ax.hexbin(pairs['spring_temp_anomaly'], pairs['onset_anomaly'], gridsize=60, mincnt=1, bins='log', cmap='viridis')
    ax.axhline(0, color='black', linestyle='--'); ax.axvline(0, color='black', linestyle='--')
    ax.set_xlabel('basin-mean spring 2 m temperature anomaly [K]'); ax.set_ylabel('basin-mean runoff onset anomaly [days]')
    f.colorbar(hb, label='log10(count)')
    print(pairs.corr().round(3))
else:
    print('no ERA5 zonal variables in the basin cube yet (run pipeline/scripts/era5_zonal.py, then reduce_partials.py)')

## Exploration: SWE, population, water volume vs onset statistics

In [ ]:
f,ax=plt.subplots(figsize=(12,7))
basins_means_gdf.plot.scatter(ax=ax,x='POPULATION',y='pct_precip_as_snow',c='runoff_onset_mad',cmap='Reds',edgecolor='black',linewidth=0.5)
# make x axis log scale
ax.set_xscale('log')

In [ ]:
f,axs=plt.subplots(nrows=5,ncols=2, figsize=(10,10), subplot_kw={'projection': ccrs.PlateCarree()}, dpi=300, layout='constrained',sharex=True, sharey=True)

for water_year, ax in zip(water_years,axs.flat):
    basins_means_gdf.plot(ax=ax,column=f'swe_anomaly_WY{water_year}',cmap='RdBu',vmin=-0.5,vmax=0.5,legend=False, transform=ccrs.PlateCarree())
    ax.set_title(f'{water_year}')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, xlocs=[-180, -120, -60, 0, 60, 120, 180], ylocs=[-60, -40, -20, 0, 20, 40, 60, 80], linestyle='--', linewidth=0.5)
    gl.top_labels=False
    gl.bottom_labels=True
    gl.right_labels=False
    gl.left_labels=True

    ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor='lightgrey')
    ax.add_feature(cfeature.OCEAN, facecolor='powderblue') # 'lightcyan'
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
basins_means_gdf.plot.scatter(ax=ax,y='swe_median',x='runoff_onset_median',c='runoff_onset_mad',cmap='Reds',vmin=0,vmax=30,alpha=1, s=10)
ax.set_ylim(0,3)
ax.set_xlabel("10-year median runoff onset date [DOWY]")
ax.set_ylabel("10-year median April 1st / October 1st SWE [m]")

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
basins_means_gdf.plot.scatter(ax=ax,y='swe_median',x='runoff_onset_mad',c='runoff_onset_median', cmap='viridis',s=10)
ax.set_ylim(0,3)
ax.set_xlabel("10-year runoff onset median absolute deivations [days]")
ax.set_ylabel("10-year median April 1st / October 1st SWE [m]")
# which basins have a high SWE and low MAD?
# which basins have a high SWE and high MAD (more dangerous)

In [ ]:
sizes = np.log(basins_means_gdf['POPULATION']+1)
sizes

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
basins_means_gdf.plot.scatter(ax=ax,y='swe_median',x='runoff_onset_mad',c='runoff_onset_median', cmap='viridis',s=5*sizes)
ax.set_ylim(0,3) # 0.1,0.5,1
ax.set_xlabel("10-year runoff onset median absolute deivations [days]")
ax.set_ylabel("10-year median April 1st / October 1st SWE [m]")
# which basins have a high SWE and low MAD?
# which basins have a high SWE and high MAD (more dangerous)

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
basins_means_gdf.plot.scatter(ax=ax,y='swe_median',x='runoff_onset_median',c='runoff_onset_mad',cmap='Reds',vmin=0,vmax=30,alpha=1, s=sizes)
ax.set_ylim(0,3)
ax.set_xlabel("10-year median runoff onset date [DOWY]")
ax.set_ylabel("10-year median April 1st / October 1st SWE [m]")

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
basins_means_gdf.plot.scatter(ax=ax,y='median_total_water_equivalent_km3',x='runoff_onset_mad',c='runoff_onset_median', cmap='viridis',s=5*sizes)
ax.set_xlabel("10-year runoff onset median absolute deivations [days]")
ax.set_ylabel("10-year median April 1st / October 1st total water equivalent [km^3]")
# which basins have a high SWE and low MAD?
# which basins have a high SWE and high MAD (more dangerous)

In [ ]:
scatter_colors = plt.cm.tab20(np.linspace(0, 1, len(water_years)))   # one colour per water year
f,ax=plt.subplots(figsize=(10,10))

for water_year, color in zip(water_years,scatter_colors):
    basins_means_gdf[basins_means_gdf['swe_median']>0.0].plot.scatter(ax=ax,y=f'swe_anomaly_WY{water_year}',x=f'runoff_onset_anomaly_WY{water_year}',color=color,s=1,alpha=1,label=water_year)



ax.axvline(x=0, color='black', linestyle='--')
ax.axhline(y=0, color='black', linestyle='--')

ax.set_ylim(-0.5,0.5)
ax.set_xlim(-30,30)

ax.set_xlabel("Runoff onset anomaly [days]")
ax.set_ylabel("SWE anomaly [m]")

ax.legend()

In [ ]:
scatter_colors = plt.cm.tab20(np.linspace(0, 1, len(water_years)))   # one colour per water year
f,ax=plt.subplots(figsize=(10,10))

for water_year, color in zip(water_years,scatter_colors):
    basins_means_gdf[basins_means_gdf['swe_median']>0.0].plot.scatter(ax=ax,y=f'total_water_equivalent_anomaly_km3_WY{water_year}',x=f'runoff_onset_anomaly_WY{water_year}',color=color,s=1,alpha=1,label=water_year)



ax.axvline(x=0, color='black', linestyle='--')
ax.axhline(y=0, color='black', linestyle='--')

#ax.set_ylim(-0.5,0.5)
ax.set_xlim(-30,30)

ax.set_xlabel("Runoff onset anomaly [days]")
ax.set_ylabel("Basin total water volume anomaly [km^3]")

ax.legend()

In [ ]:
# Define water years
water_years = [int(y) for y in config.water_years]

# Create empty list to store data rows
data_rows = []

# Filter to basins with SWE median > 0.0 as in your previous analyses
filtered_data = basins_means_gdf[basins_means_gdf['swe_median'] > 0.0]

# Loop through each basin
for idx, basin_row in filtered_data.iterrows():
    basin_id = basin_row['PFAF_ID']  # Using PFAF_ID as basin identifier
    
    # For each water year, extract the anomaly values
    for year in water_years:
        runoff_col = f'runoff_onset_anomaly_WY{year}'
        swe_col = f'swe_anomaly_WY{year}'
        
        # Check if columns exist and values are not NaN
        if runoff_col in filtered_data.columns and swe_col in filtered_data.columns:
            runoff_anomaly = basin_row[runoff_col] 
            swe_anomaly = basin_row[swe_col]
            
            # Only include rows where both values are valid
            if pd.notna(runoff_anomaly) and pd.notna(swe_anomaly):
                data_rows.append({
                    'basin': basin_id,
                    'runoff_onset_anomaly': runoff_anomaly,
                    'swe_anomaly': swe_anomaly,
                    'water_year': year
                })

# Create dataframe from collected data
anomalies_df = pd.DataFrame(data_rows)
anomalies_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# Create figure for KDE plot
fig_kde, ax_kde = plt.subplots(figsize=(10, 10))

# Create the KDE plot
sns.kdeplot(
    data=anomalies_df,
    x='runoff_onset_anomaly',
    y='swe_anomaly',
    fill=True,
    cmap="viridis",
    levels=30,
    ax=ax_kde
)

# Calculate line of best fit
slope, intercept, r_value, p_value, std_err = stats.linregress(
    anomalies_df['runoff_onset_anomaly'].dropna(), 
    anomalies_df['swe_anomaly'].dropna()
)

# Add line of best fit to KDE plot
x_line = np.linspace(-30, 30, 100)
y_line = slope * x_line + intercept
ax_kde.plot(x_line, y_line, color='red', linestyle='--', linewidth=2, 
            label=f'y = {slope:.4f}x + {intercept:.4f} (r = {r_value:.2f})')

# Add reference lines
ax_kde.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax_kde.axhline(y=0, color='black', linestyle='--', alpha=0.5)

# Set limits and labels
ax_kde.set_ylim(-0.3, 0.3)
ax_kde.set_xlim(-30, 30)
ax_kde.set_xlabel('Runoff Onset Anomaly [days]')
ax_kde.set_ylabel('SWE Anomaly [m]')
ax_kde.set_title('KDE Plot of SWE vs Runoff Onset Anomalies')
ax_kde.legend()

# Create figure for hexbin plot
fig_hex, ax_hex = plt.subplots(figsize=(10, 10))

# Create the hexbin plot
hb = ax_hex.hexbin(
    anomalies_df['runoff_onset_anomaly'], 
    anomalies_df['swe_anomaly'], 
    gridsize=100, 
    cmap='viridis', 
    mincnt=1,
    bins='log'  # Use log scale for better visualization
)

# Add line of best fit to hexbin plot
ax_hex.plot(x_line, y_line, color='red', linestyle='--', linewidth=2,
            label=f'y = {slope:.4f}x + {intercept:.4f} (r = {r_value:.2f})')

# Add reference lines
ax_hex.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax_hex.axhline(y=0, color='black', linestyle='--', alpha=0.5)

# Set limits and labels
ax_hex.set_ylim(-0.3, 0.3)
ax_hex.set_xlim(-30, 30)
ax_hex.set_xlabel('Runoff Onset Anomaly [days]')
ax_hex.set_ylabel('SWE Anomaly [m]')
ax_hex.set_title('Hexbin Plot of SWE vs Runoff Onset Anomalies')
ax_hex.legend()

# Add colorbar to hexbin plot
cb = fig_hex.colorbar(hb, ax=ax_hex)
cb.set_label('log10(count)')

plt.tight_layout()
plt.show()

In [ ]:
scatter_colors = plt.cm.tab20(np.linspace(0, 1, len(water_years)))   # one colour per water year
f,ax=plt.subplots(figsize=(10,10))

for water_year, color in zip(water_years,scatter_colors):
    basins_means_gdf.plot.scatter(ax=ax,y=f'swe_pct_norm_WY{water_year}',x=f'runoff_onset_anomaly_WY{water_year}',color=color,s=1, alpha=1)

ax.set_ylim(0,1000)
ax.set_xlim(-60,60)

# ax.set_ylim(0)
# ax.set_xlim(-30,30)

ax.axvline(x=0, color='black', linestyle='--')
ax.axhline(y=100, color='black', linestyle='--')

In [ ]:
# Define water years
water_years = [int(y) for y in config.water_years]

# Create empty list to store data rows
data_rows = []

# Filter to basins with SWE median > 0.0 as in your previous analyses
filtered_data = basins_means_gdf[basins_means_gdf['swe_median'] > 0.0]

# Loop through each basin
for idx, basin_row in filtered_data.iterrows():
    basin_id = basin_row['PFAF_ID']  # Using PFAF_ID as basin identifier
    
    # For each water year, extract the anomaly values
    for year in water_years:
        runoff_col = f'runoff_onset_anomaly_WY{year}'
        swe_col = f'total_water_equivalent_anomaly_km3_WY{year}'
        
        # Check if columns exist and values are not NaN
        if runoff_col in filtered_data.columns and swe_col in filtered_data.columns:
            runoff_anomaly = basin_row[runoff_col] 
            swe_anomaly = basin_row[swe_col]
            
            # Only include rows where both values are valid
            if pd.notna(runoff_anomaly) and pd.notna(swe_anomaly):
                data_rows.append({
                    'basin': basin_id,
                    'runoff_onset_anomaly': runoff_anomaly,
                    'basin_total_water_volume_anomaly': swe_anomaly,
                    'water_year': year
                })

# Create dataframe from collected data
anomalies_df = pd.DataFrame(data_rows)
anomalies_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# Create figure for KDE plot
fig_kde, ax_kde = plt.subplots(figsize=(10, 10))

# Create the KDE plot
sns.kdeplot(
    data=anomalies_df,
    x='runoff_onset_anomaly',
    y='basin_total_water_volume_anomaly',
    fill=True,
    cmap="viridis",
    levels=30,
    ax=ax_kde
)

# Calculate line of best fit
slope, intercept, r_value, p_value, std_err = stats.linregress(
    anomalies_df['runoff_onset_anomaly'].dropna(), 
    anomalies_df['basin_total_water_volume_anomaly'].dropna()
)

# Add line of best fit to KDE plot
x_line = np.linspace(-30, 30, 100)
y_line = slope * x_line + intercept
ax_kde.plot(x_line, y_line, color='red', linestyle='--', linewidth=2, 
            label=f'y = {slope:.4f}x + {intercept:.4f} (r = {r_value:.2f})')

# Add reference lines
ax_kde.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax_kde.axhline(y=0, color='black', linestyle='--', alpha=0.5)

# Set limits and labels
#ax_kde.set_ylim(-0.3, 0.3)
ax_kde.set_xlim(-30, 30)
ax_kde.set_xlabel('Runoff Onset Anomaly [days]')
ax_kde.set_ylabel('Basin total water volume anomaly [km^3]')
ax_kde.set_title('KDE Plot of Basin Total Water Volume Anomalies vs Runoff Onset Anomalies')
ax_kde.legend()

# Create figure for hexbin plot
fig_hex, ax_hex = plt.subplots(figsize=(10, 10))

# Create the hexbin plot
hb = ax_hex.hexbin(
    anomalies_df['runoff_onset_anomaly'], 
    anomalies_df['basin_total_water_volume_anomaly'], 
    gridsize=100, 
    cmap='viridis', 
    mincnt=1,
    bins='log'  # Use log scale for better visualization
)

# Add line of best fit to hexbin plot
ax_hex.plot(x_line, y_line, color='red', linestyle='--', linewidth=2,
            label=f'y = {slope:.4f}x + {intercept:.4f} (r = {r_value:.2f})')

# Add reference lines
ax_hex.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax_hex.axhline(y=0, color='black', linestyle='--', alpha=0.5)

# Set limits and labels
#ax_hex.set_ylim(-0.3, 0.3)
ax_hex.set_xlim(-30, 30)
ax_hex.set_xlabel('Runoff Onset Anomaly [days]')
ax_hex.set_ylabel('Basin total water volume anomaly [km^3]')
ax_hex.set_title('Hexbin Plot of Basin Total Water Volume Anomalies vs Runoff Onset Anomalies')
ax_hex.legend()

# Add colorbar to hexbin plot
cb = fig_hex.colorbar(hb, ax=ax_hex)
cb.set_label('log10(count)')

plt.tight_layout()
plt.show()